In [1]:
import os
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [ ]:
DATASET_DIR = Path(r"D:\Datasets\<dataset_name>")

print(DATASET_DIR.exists())

True


In [ ]:
for item in sorted(DATASET_DIR.iterdir()):
    print(item.name)

In [4]:
session = DATASET_DIR / "Session1"

for item in sorted(session.iterdir()):
    print(item.name)

dialog
sentences


In [5]:
# Count sessions
sessions = sorted(DATASET_DIR.glob("Session*"))

print(f"Number of sessions: {len(sessions)}")

Number of sessions: 5


In [6]:
def tree(path, depth=2):

    for root, dirs, files in os.walk(path):

        level = root.replace(str(path), "").count(os.sep)

        if level > depth:
            continue

        indent = "    " * level

        print(f"{indent}{Path(root).name}/")

        subindent = "    " * (level + 1)

        for f in files[:5]:
            print(f"{subindent}{f}")

In [ ]:
tree(session)

In [8]:
# Count audio files
wav_files = list(DATASET_DIR.rglob("*.wav"))

len(wav_files)

10190

In [9]:
# Count transcriptions
txt_files = list(DATASET_DIR.rglob("*.txt"))

len(txt_files)

31610

In [10]:
# Count emotion labels
emo_files = list(DATASET_DIR.rglob("*.txt"))

emo_eval = [f for f in emo_files if "EmoEvaluation" in str(f)]

len(emo_eval)

1101

In [ ]:
# Inspect one emotion annotation
sample = emo_eval[0]

print(sample)

print("-"*80)

with open(sample, encoding="utf8") as f:

    for i in range(25):
        print(f.readline().rstrip())

In [ ]:
# Inspect one transcription
transcripts = list(DATASET_DIR.rglob("*.txt"))

trans = [t for t in transcripts if "transcriptions" in str(t)][0]

print(trans)

print()

with open(trans, encoding="utf8") as f:

    for i in range(15):
        print(f.readline().rstrip())

In [ ]:
# Explore audio 
import librosa

audio = wav_files[0]

signal, sr = librosa.load(audio, sr=None)

print(audio.name)
print(signal.shape)
print(sr)
print(len(signal)/sr)

In [ ]:
# Session stats
for s in sessions:

    wavs = list(s.rglob("*.wav"))

    print(f"{s.name}: {len(wavs)} wav files")

##### Data Parser

In [15]:
import re
from pathlib import Path

import pandas as pd
import pyarrow as pa
from tqdm.auto import tqdm

In [16]:
# Helper regrex
TRANSCRIPT_PATTERN = re.compile(
    r"(\S+)\s+\[(.*?)\]:\s+(.*)"
)

EMOTION_PATTERN = re.compile(
    r"\[(.*?) - (.*?)\]\s+(\S+)\s+(\w+)"
)

In [17]:
# Parse one transcript 
def parse_transcript(path: Path):

    utterances = {}

    with open(path, encoding="utf8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            m = TRANSCRIPT_PATTERN.match(line)

            if m is None:
                continue

            utt_id = m.group(1)

            times = m.group(2)

            transcript = m.group(3)

            start, end = times.split("-")

            utterances[utt_id] = {
                "start": float(start),
                "end": float(end),
                "transcript": transcript,
            }

    return utterances

In [18]:
# Parse emotion
def parse_emotions(path: Path):

    emotions = {}

    with open(path, encoding="utf8") as f:

        for line in f:

            line = line.strip()

            m = EMOTION_PATTERN.match(line)

            if m is None:
                continue

            start = float(m.group(1))

            end = float(m.group(2))

            utt_id = m.group(3)

            emotion = m.group(4)

            emotions[utt_id] = {
                "emotion": emotion,
                "start": start,
                "end": end,
            }

    return emotions

In [19]:
# Build metadata 
rows = []

for session in tqdm(sessions):

    transcript_dir = session / "dialog" / "transcriptions"

    emotion_dir = session / "dialog" / "EmoEvaluation"

    sentence_wav = session / "sentences" / "wav"

    for transcript_file in transcript_dir.glob("*.txt"):

        dialogue = transcript_file.stem

        transcript_data = parse_transcript(transcript_file)

        emotion_data = parse_emotions(
            emotion_dir / f"{dialogue}.txt"
        )

        for utt_id, text_info in transcript_data.items():

            wav_path = (
                sentence_wav
                / dialogue
                / f"{utt_id}.wav"
            )

            emotion_info = emotion_data.get(utt_id)

            if emotion_info is None:
                continue

            rows.append(
                {
                    "utterance_id": utt_id,
                    "session": session.name,
                    "dialogue": dialogue,
                    "speaker": utt_id.split("_")[-1][0],
                    "emotion": emotion_info["emotion"],
                    "start": text_info["start"],
                    "end": text_info["end"],
                    "duration": text_info["end"] - text_info["start"],
                    "transcript": text_info["transcript"],
                    "wav_path": str(wav_path),
                    "wav_exists": wav_path.exists(),
                }
            )

100%|██████████| 5/5 [00:00<00:00,  9.92it/s]


In [ ]:
metadata = pd.DataFrame(rows)

metadata.head()

In [21]:
print(metadata.shape)

print()

print(metadata.isna().sum())

print()

print(metadata.wav_exists.value_counts())

(10039, 11)

utterance_id    0
session         0
dialogue        0
speaker         0
emotion         0
start           0
end             0
duration        0
transcript      0
wav_path        0
wav_exists      0
dtype: int64

wav_exists
True    10039
Name: count, dtype: int64


In [22]:
# Emotion distrib
metadata.emotion.value_counts()

emotion
xxx    2507
fru    1849
neu    1708
ang    1103
sad    1084
exc    1041
hap     595
sur     107
fea      40
oth       3
dis       2
Name: count, dtype: int64

In [23]:
# Duration stats
metadata.duration.describe()

count    10039.000000
mean         4.460109
std          3.064676
min          0.585000
25%          2.330000
50%          3.520000
75%          5.710000
max         34.138800
Name: duration, dtype: float64

In [24]:
# Speaker distrib
metadata.speaker.value_counts()

speaker
M    5239
F    4800
Name: count, dtype: int64

In [ ]:
# Rand samples
metadata.sample(10)[
    [
        "utterance_id",
        "emotion",
        "transcript",
        "duration",
    ]
]

In [26]:
# Missing wavs
missing = metadata.loc[~metadata.wav_exists]

missing

,utterance_id,session,dialogue,speaker,emotion,start,end,duration,transcript,wav_path,wav_exists


In [27]:
# Duplicate IDs
metadata.utterance_id.duplicated().sum()

np.int64(0)

In [ ]:
# Export
output = DATASET_DIR / "metadata.parquet"

metadata.to_parquet(output, index=False)

print(output)

In [29]:
metadata.to_csv(
    DATASET_DIR / "metadata.csv",
    index=False,
)

In [ ]:
# Avg transcript length
metadata["num_words"] = (
    metadata["transcript"]
    .str.split()
    .str.len()
)

metadata["num_words"].describe()

In [ ]:
# Emotion × Duration
metadata.groupby("emotion")["duration"].describe()